In [ ]:
watch -n 1 nvidia-smi

In [ ]:
ls /dev/nvidia*

In [ ]:
import os
os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"

from unsloth import FastLanguageModel
import torch

# 1. 配置参数：切换到 Qwen2.5-7B
# Qwen2.5 是目前性价比极高的选择，7B 版本在 3090 上跑 4-bit 飞快
model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit" 
max_seq_length = 2048 

# 2. 加载千问模型
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 3. 准备推理测试
FastLanguageModel.for_inference(model)

# 构造一个测试：普通话转粤语
messages = [
    {"role": "system", "content": "你是一个地道的粤语翻译助手。"},
    {"role": "user", "content": "请把这句话翻译成粤语：不过又真像中饱私囊挺深"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

print("\n🚀 Qwen2.5 正在生成测试回答...")
outputs = model.generate(input_ids = inputs, max_new_tokens = 128)
response = tokenizer.batch_decode(outputs)

# 打印结果（过滤掉模板标签）
print("-" * 30)
print(response[0].split("assistant\n")[-1].replace("<|im_end|>", "").strip())
print("-" * 30)
print("✅ 千问模型点火成功！效果满意吗？")

In [ ]:
import os
os.environ["UNSLOTH_SKIP_TORCHVISION_CHECK"] = "1"
from unsloth import FastLanguageModel
import torch

# 1. 路径指向你最新的 4000 步存档
model_path = "outputs_yue_qwen/checkpoint-10000"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)



In [3]:
# 2. 这里的测试题我选了一句带有“地道语气”和“复杂情绪”的话
test_text = "7月18號，喺廣州體育館舉辦《著迷·陳潔儀》巡回演唱會廣州站。"

prompt = f"<|im_start|>system\n你是一个地道的粤语翻译助手。<|im_end|>\n<|im_start|>user\n{test_text}<|im_end|>\n<|im_start|>assistant\n"
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# 3. 生成翻译
outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
result = tokenizer.decode(outputs[0], skip_special_tokens = True)

print("\n" + "="*30)
print("【10000步面试结果】")
print(result.split("assistant")[-1].strip())
print("="*30)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



【10000步面试结果】
7月18日，在广州体育馆举办《着迷·陈洁仪》巡回演唱会广州站。


In [7]:
import os
# 设置国内镜像站环境变量
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

from huggingface_hub import snapshot_download

try:
    snapshot_download(
        repo_id = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
        local_dir = "./base_model",
        local_dir_use_symlinks = False,
        ignore_patterns = ["*.msgpack", "*.h5", "*.ot"],
    )
    print("✅ 镜像站下载成功！")
except Exception as e:
    print(f"❌ 还是不行：{e}")

Fetching 11 files: 100%|██████████| 11/11 [00:07<00:00,  1.40it/s]

✅ 镜像站下载成功！
